## Reproduce analyses, plots, and tables for Salz et al., 2026
This Jupyter notebook can be used to reproduce all figures in the soil half-life paper

In [113]:
import pandas as pd
import os
import matplotlib 
import matplotlib.pyplot as plt 
import seaborn as sns
from copy import deepcopy
import numpy as np
from rdkit import Chem
from rdkit.Chem.Draw import rdMolDraw2D
from scipy.stats import norm, spearmanr, pearsonr, wilcoxon
from sklearn.metrics import root_mean_squared_error

from IPython.core.pylabtools import figsize
from sklearn.metrics import root_mean_squared_error
from sklearn.metrics import r2_score
from pepper_lab.pepper import Pepper
from pepper_lab.datastructure import DataStructure
from pepper_lab.datastructuresoil import DataStructureSoil
from pepper_lab.bayesian import Bayesian


In [114]:
# plotting settings
context = 'paper'  # other option: presentation
color_palette_3 = ["#428aa3", "#c3618b", "#7a8838"]
color_palette_1 = ["#63baab","#006b5e", "#ff7f0e"]
color_palette_gradient = ["#428aa3", "#4b87b0", "#6581b8", "#8579b6", "#a46eaa", "#b96696",
                                "#c5627e", "#c86365", "#bf6b4c", "#ae753a", "#968032", "#7a8838"]
color_palette_models = ['#5ab7e1', '#0082aa', '#fff189', '#ae7a0d', '#9fe1d5', '#003c34', '#f380f0', '#5e0062', '#d4a375', '#9b6f44', '#d5cabd', '#50434f']
color_palette_descriptors = ['#f380f0', '#810082', '#74b8ac', '#005249', '#a2acbd', '#2c4a6e', '#ffedcb', '#dda11d']
color_palette_data_sets = ['#60baae', '#2f4858', '#006b61', '#2f4858', '#92a19f', '#554516', '#b49205']

# variable names
mu_mean = r'$\mathrm{\mu_{mean}}$'

mu_std = r'$\mathrm{\mu_{std}}$'

Define paths

In [5]:
print(os.path.abspath(os.sep))
current_path = os.getcwd()
pep = Pepper(pepper_data_location=os.path.join(current_path, "..", ".."))
pep.build_directory_structure('soil_paper_output') # where all the figures for the paper will be saved
save_output_path = os.path.join(pep.get_data_directory(), 'soil_paper_output')

/


Prepare pepper object and load soil data

In [12]:
pep.set_tag('all_data')
pep.set_setup_name('all_data')
pep.set_data_type('soil')
pep.set_target_variable_name('logDT50_mean')
pep.set_target_variable_std_name('logDT50_std')
pep.set_smiles_name('SMILES')
pep.set_compound_name('compound_name')
soil = DataStructureSoil(pep)
soil.curate_annotate(from_csv=True, from_paper=True)
soil.reduce_for_modelling(from_csv=False)


############# Reduce data set ############# 
Data frame size:  6309
Data frame size:  874
Compound data file saved to /Users/jasmin/pepper/scripts/../../pepper_data/data_structure/cpd_data_soil_all_data.tsv

############# Manual curation ############# 
Original number of compounds: 874
Organic substances: 869
Data set without composite mixtures: 867
Number of duplicated cropped canonical SMILES without stereo information: 0


### S1.1 Bayesian Inference

Figure S2: Distribution of Bayesian inferred log half-life means (μmean) and mean uncertainties (μstd)

In [ ]:
soil_data = deepcopy(soil.cpd_data)
soil_data['logDT50_mean'] = soil_data['DT50_log_bayesian_mean']
soil_data['logDT50_std'] = soil_data['DT50_log_bayesian_std']

sns.set_style('white')
sns.set_context(context)
fig, axs = plt.subplots(1,2, figsize=(12, 4))
sns.kdeplot(soil_data['logDT50_mean'], fill=True, color='#c3618b', alpha=0.6, ax=axs[0])
muexperimental = soil_data['logDT50_mean'].mean()
stdexperimental = soil_data['logDT50_mean'].std()
axs[0].text(3, 0.45, f'mean = {muexperimental:.2f}\nstd = {stdexperimental:.2f}', fontsize=12, verticalalignment='top', horizontalalignment='left')

sns.kdeplot(soil_data['logDT50_std'], fill=True, color='#c3618b', alpha=0.6, ax=axs[1])

mstdexperimental = soil_data['logDT50_std'].mean()
stdstdexperimental = soil_data['logDT50_std'].std()
axs[1].text(0.67, 3.5, f'mean = {mstdexperimental:.2f}\nstd = {stdstdexperimental:.2f}', fontsize=12)

axs[0].set_xlabel('$\mathrm{\mu_{mean}}$', fontsize=12)
axs[1].set_xlabel('$\mathrm{\mu_{std}}$', fontsize=12)
axs[1].set_ylabel('')
plt.tight_layout()
plt.savefig(os.path.join(save_output_path, "data_distribution_soil_logDT50_mean_std.pdf"), dpi=600)
plt.show()

print("range of logDT50_mean:", soil_data['logDT50_mean'].min(), "-", soil_data['logDT50_mean'].max())
print("range of logDT50_std:", soil_data['logDT50_std'].min(), "-", soil_data['logDT50_std'].max())

Figure S3: Dependence of uncertainty of μmean on the number of experimental values per compound

In [ ]:
plt.figure(figsize=(10, 6))
sns.set_style('whitegrid')
sns.scatterplot(data=soil_data, x='DT50_count', y='logDT50_std', s=20, alpha=1, legend=False)
plt.xlabel('Number of reported $\mathrm{DT_{50}}$ per compound', fontsize=12)
plt.ylabel('$\mathrm{\mu_{std}}$', fontsize=12)
plt.tight_layout()
# plt.grid(True)
plt.xlim(0, soil_data['DT50_count'].max() + 1)
plt.savefig(os.path.join(save_output_path, "dt50_count_vs_logDT50_std.pdf"), dpi=600)
plt.show()

### S1.2 REACH class probabilities
Calculate persistenc probabilities

In [32]:
soil_data["Persistence_without_std"] = "nP"

def calculate_prediction_probabilities(mu, sigma, T_P, T_vP):
    log_T_P = np.log10(T_P)
    log_T_vP = np.log10(T_vP)

    zP = (log_T_P - mu) / sigma
    zvP = (log_T_vP - mu) / sigma

    p_nP = norm.cdf(zP)
    p_P = 1 - norm.cdf(zP) 
    p_vP = 1 - norm.cdf(zvP)

    return p_nP, p_P, p_vP

p_nP, p_P, p_vP = calculate_prediction_probabilities(soil_data["logDT50_mean"], soil_data["logDT50_std"], 120, 180)
soil_data['p(nP)'] = p_nP
soil_data['p(P)'] = p_P
soil_data['p(vP)'] = p_vP

soil_data.loc[soil_data["logDT50_mean"] > np.log10(180), "Persistence_without_std"] = "vP"

soil_data.loc[(soil_data["logDT50_mean"] <= np.log10(180)) & (soil_data["logDT50_mean"]>= np.log10(120)), "Persistence_without_std"] = "P"

soil_data.sort_values(by="Persistence_without_std", inplace=True)


soil_data["Persistence_with_std"] = "nP"

soil_data.loc[soil_data["logDT50_mean"] + soil_data["logDT50_std"] * 1.96 > np.log10(180), "Persistence_with_std"] = "vP"

soil_data.loc[(soil_data["logDT50_mean"] + soil_data["logDT50_std"] * 1.96 <= np.log10(180)) & (soil_data["logDT50_mean"] +soil_data["logDT50_std"] * 1.96 >= np.log10(120)), "Persistence_with_std"] = "P"

soil_data.sort_values(by="Persistence_with_std", inplace=True)



# Plotting the regulatory tresholds
persistence_order = ['nP', 'P', 'vP']
soil_data['Persistence_without_std'] = pd.Categorical(soil_data['Persistence_without_std'], categories=persistence_order, ordered=True)
soil_data['Persistence_with_std'] = pd.Categorical(soil_data['Persistence_with_std'], categories=persistence_order, ordered=True)


Define plotting functions

In [37]:
from matplotlib.offsetbox import OffsetImage, AnnotationBbox
import io
import PIL.Image as Image

def smiles_to_img_array(smile, size=(400, 400)):
    """Return a NumPy array with a RDKit drawing of the SMILES."""
    mol = Chem.MolFromSmiles(smile)
    drawer = rdMolDraw2D.MolDraw2DCairo(size[0], size[1])
    opts = drawer.drawOptions()
    opts.useBWAtomPalette()
    drawer.DrawMolecule(mol)
    drawer.FinishDrawing()
    png_bytes = drawer.GetDrawingText()
    img = Image.open(io.BytesIO(png_bytes))
    return np.array(img)

def plot_regulatory_combined(soil_data):
    
    # --- Helper function to draw each panel ---
    def plot_panel(ax, soil_data, with_std):

        if with_std:
            soil_data_sorted = soil_data.sort_values(
                by=['Persistence_with_std', 'logDT50_mean']
            )
            name = 'Persistence_with_std'
        else:
            soil_data_sorted = soil_data.sort_values(
                by=['Persistence_without_std', 'logDT50_mean']
            )
            name = 'Persistence_without_std'

        soil_data_sorted = soil_data_sorted.copy()

        # keep ID for printing etc, but we use numeric x for plotting
        soil_data_sorted["xpos"] = np.arange(len(soil_data_sorted))

        sns.set_theme(style="whitegrid", font_scale=1.5)
        sns.set_style("ticks")

        custom_palette = {'nP': 'green', 'P': 'orange', 'vP': 'red'}

        # Scatterplot
        sns.scatterplot(
            ax=ax,
            data=soil_data_sorted,
            x="xpos",
            y="logDT50_mean",
            hue=name,
            palette=custom_palette,
            s=12,
            alpha=0.6
        )

        # Error bars
        for _, row in soil_data_sorted.iterrows():
            ax.errorbar(
                x=row["xpos"],
                y=row["logDT50_mean"],
                yerr=row["logDT50_std"] * 1.96,
                fmt="none",
                ecolor="gray",
                elinewidth=1,
                capsize=2,
                alpha=0.2
            )

        # Threshold lines
        ax.axhline(y=np.log10(180), color='red', linestyle='--', linewidth=1.5)
        ax.axhline(y=np.log10(120), color='orange', linestyle='--', linewidth=1.5)

        # Text labels & legend
        if not with_std:
            ax.text(0.5, np.log10(180)+0.2, "180 days (REACH)", color="red", fontsize=22)
            ax.text(0.5, np.log10(120)-0.4, "120 days (REACH)", color="orange", fontsize=22)



        # compute counts & percentages
        total_count = len(soil_data_sorted)
        counts = soil_data_sorted[name].value_counts().reindex(persistence_order, fill_value=0)

        # get current legend entries
        handles, labels = ax.get_legend_handles_labels()

        # build new labels with percentages
        new_labels = []
        for lab in labels:
            if lab in counts.index:
                count = counts[lab]
                pct = count / total_count * 100
                new_labels.append(f"{lab} ({pct:.1f}%)")
            else:
                new_labels.append(lab)

        # draw legend
        ax.legend(
            handles,
            new_labels,
            title="Persistence",
            fontsize=15
        )

        # --- vertical separators + % labels near top (black only) ---
        classes = soil_data_sorted[name].to_numpy()
        n = len(classes)

        # find contiguous class blocks
        blocks = []
        start = 0
        for i in range(1, n):
            if classes[i] != classes[i - 1]:
                blocks.append((classes[start], start, i - 1))
                start = i
        blocks.append((classes[start], start, n - 1))

        # draw separators
        for _, s, _ in blocks[1:]:
            ax.axvline(
                x=s - 0.5,
                color="black",
                linewidth=1.0,
                alpha=0.25,
                zorder=0
            )

        # text transform: x=data, y=axes
        trans = ax.get_xaxis_transform()

        # add percentages centered over each block
        for cat, s, e in blocks:
            mid = (s + e) / 2
            pct = counts[cat] / total_count * 100

            ax.text(
                mid,
                0.98,                 # near top edge
                f"{pct:.1f}%",
                transform=trans,
                ha="center",
                va="top",
                fontsize=12,
                clip_on=False,
                zorder=20
            )

        

        ax.set_xticks([])
        ax.set_ylabel(mu_mean, fontsize=22)
        ax.tick_params(axis='y', labelsize=16)
        ax.set_xlabel("Compounds", fontsize=22)

        return name, soil_data_sorted

    # --- Create the combined figure ---
    fig, axes = plt.subplots(1, 2, figsize=(18, 8), sharey=True)

    # Left: without std
    name1, sorted1 = plot_panel(axes[0], soil_data, with_std=False)

    # Right: with std
    name2, sorted2 = plot_panel(axes[1], soil_data, with_std=True)

    # --- pick the 2 highlighted compounds (P, vP with smallest logDT50_mean) ---
    highlight_rows = []
    for cat in ["P", "vP"]:
        subset = sorted2[sorted2[name2] == cat]
        if subset.empty:
            continue
        row_min = subset.sort_values("logDT50_mean").iloc[0]
        highlight_rows.append(row_min)

    if highlight_rows:
        highlight_df = pd.DataFrame(highlight_rows)

        print("\nHighlighted compounds:")
        print(highlight_df[["ID", "logDT50_mean", "logDT50_std", name2]])

        # 1) keep your circle highlight on the right panel
        axes[1].scatter(
            x=highlight_df["xpos"],
            y=highlight_df["logDT50_mean"],
            s=300,
            facecolor="none",
            edgecolor="black",
            linewidth=2.5,
            zorder=10
        )

        # 2) add the *structures* anywhere on the right axis (axes coords)
        #    e.g. top-right corner, stacked
        for i, (_, row) in enumerate(highlight_df.iterrows()):
            smile = row["SMILES"]
            name = row["compound_name"]
            if name == 'Unknown AE-41  (tentatively identified as SYN505866)': # replace long name
                name = 'SYN505866'
            img_arr = smiles_to_img_array(smile, size=(500, 500))

            imagebox = OffsetImage(img_arr, zoom=0.3)  # widen slightly for horizontal layout

            # place horizontally: x increases with i, y fixed (near top)
            x_pos = 0.4 + i * 0.32   # controls spacing; adjust to fit
            y_pos = 0.3              # fixed height

            ab = AnnotationBbox(
                imagebox,
                (x_pos, y_pos),
                xycoords=axes[1].transAxes,   # <-- axes coordinates
                frameon=False,
                box_alignment=(0.0, 1.0)
            )
            axes[1].add_artist(ab)

            text_x = x_pos + (0.14 if i == 0 else 0.0)
            axes[1].text(
            text_x,
            y_pos - 0.27,        # vertical offset below image (adjust as needed)
            name,
            transform=axes[1].transAxes,
           
            fontsize=11,
            wrap=True            # useful if names are long
            )

            axes[1].annotate(
            "",
            xy=(x_pos + 0.15, y_pos),              # near structure (axes coords)
            xycoords=axes[1].transAxes,
            xytext=(row["xpos"], row["logDT50_mean"]),  # circled point (data coords)
            textcoords=axes[1].transData,
            arrowprops=dict(
                arrowstyle="-",
                linewidth=1.8,
                color="black",
                alpha=0.8
            ),
            zorder=12
        )
            

    plt.tight_layout()
    save_path = os.path.join(save_output_path, "soil_logDT50_mean_by_compound_id_combined_structures.pdf")
    print("\nSaving figure to: ", save_path)
    plt.savefig(
        save_path,
        dpi=600
    )
    plt.show()


    # Print counts for both
    print("\nCounts per persistence category (without std):")
    print(sorted1[name1].value_counts())

    print("\nCounts per persistence category (with std):")
    print(sorted2[name2].value_counts())


def persistence_probs(mu, sigma, T_P=120, T_vP=180):
    log_T_P = np.log10(T_P)
    log_T_vP = np.log10(T_vP)
    zP = (log_T_P - mu) / sigma
    zvP = (log_T_vP - mu) / sigma
    p_nP = norm.cdf(zP)
    p_P = 1 - norm.cdf(zP) 
    p_vP = 1 - norm.cdf(zvP)
    return p_nP, p_P, p_vP

Figure 2: REACH persistence classification

In [ ]:
plot_regulatory_combined(soil_data)
soil_data.to_csv(os.path.join(save_output_path, "cpd_data_with_P_probabilities.csv"))

### S1.3 Sensitivity analysis

Select example compounds from Hafner et al. (2023) for sensitivity analysis: 

In [11]:
selected_compounds = ['methyl-2-butyl sulfone', 'trans Dodemorph', 'Tribenuron-methyl', 'Dazomet', 'Quizalofop-P-tefuryl', 'Triazine amine (IN-A4098)', 'Butralin', 'Fluazifop-P-butyl R isomer', 'X696476']

selected_data = soil.full_data[soil.full_data[pep.compound_name].isin(selected_compounds)]
selected_data = selected_data[[pep.compound_name, pep.id_name, 'DT50_log', 'DT50_count', 
                               'halflife_comment', 'DT50_log_bayesian_mean', 
                               'DT50_log_bayesian_mean_std', 'DT50_log_bayesian_std']]
selected_data

,compound_name,ID,DT50_log,DT50_count,halflife_comment,DT50_log_bayesian_mean,DT50_log_bayesian_mean_std,DT50_log_bayesian_std
852,Butralin,122,3.289812,1,"reported, calculation by RMS, only partly usab...",3.78,1.10,0.50
853,methyl-2-butyl sulfone,123,0.653213,1,"reported in the rate part of the dossier, reca...",0.72,0.55,0.47
1189,Triazine amine (IN-A4098),161,1.595496,59,"reported, linear first-order,no further inform...",2.07,0.06,0.48
1190,Triazine amine (IN-A4098),161,1.462398,59,"reported, linear first-order,no further inform...",2.07,0.06,0.48
1191,Triazine amine (IN-A4098),161,1.350248,59,"reported, linear first-order,no further inform...",2.07,0.06,0.48
...,...,...,...,...,...,...,...,...
5984,Triazine amine (IN-A4098),161,1.462398,59,-,2.07,0.06,0.48
6231,X696476,869,3.000000,4,"RMS suggestion as default value, since no degr...",4.29,0.95,0.46
6232,X696476,869,3.000000,4,"RMS suggestion as default value, since no degr...",4.29,0.95,0.46
6233,X696476,869,3.000000,4,"RMS suggestion as default value, since no degr...",4.29,0.95,0.46


In [12]:
sensitivity = DataStructure(pep)
selected_data['comment'] = sensitivity.process_comment_list(selected_data['halflife_comment'])
sensitivity.full_data = selected_data

Define function for modifying priors

In [13]:
def varying_priors(data_object, prior_name, prior_range, prior_default_dict):
    df_prior = pd.DataFrame()
    for prior in prior_range:
        for seed in [1,2,3]:
            new_df = deepcopy(data_object.full_data)
            sensitivity.set_random_state(seed)
            prior_default_dict[prior_name] = prior
            bmean, bstd, bmeanstd = data_object.get_bayesian_stats(target_variable='DT50_log', comment='comment',
                                       mu_mean=prior_default_dict['mu_mean'], mu_std=prior_default_dict['mu_std'], 
                                       sigma_mean=prior_default_dict['sigma_mean'], sigma_std=prior_default_dict['sigma_std'],
                                       sigma_lower_limit=prior_default_dict['sigma_lower_limit'])
            new_df[f'logDT50_mean'] = bmean
            new_df[f'logDT50_expvar'] = bstd
            new_df[f'logDT50_uncert'] = bmeanstd
            new_df[f'random_state'] = [seed]*len(bmean)
            new_df[prior_name] = [prior]*len(bmean)
            new_df.drop_duplicates(subset=['ID'], inplace=True)
            p_nP, p_P, p_vP = persistence_probs(new_df[f'logDT50_mean'],new_df[f'logDT50_uncert'])
            new_df[f'p(nP)'] = p_nP
            new_df[f'p(P)'] = p_P
            new_df[f'p(vP)'] = p_vP
            
            if df_prior.empty:
                df_prior = new_df
            else:
                df_prior = pd.concat([df_prior, new_df], sort=False)
    df_prior.drop(columns=['DT50_log', 'halflife_comment'])
    df_prior.to_csv(os.path.join(save_output_path, f'varying_{prior_name}.csv'))



Run sensitivity analysis

In [ ]:
# original settings:
prior_default_dict = {'mu_mean': 1, 'mu_std': 2, 'sigma_mean': 0.4, 'sigma_std': 0.4, 'sigma_lower_limit': 0.2}
# varying mu_mean
varying_priors(data_object=sensitivity, prior_name='mu_mean', prior_range=[0.5, 1, 1.5], prior_default_dict=prior_default_dict)
# varying mu_std
varying_priors(data_object=sensitivity, prior_name='mu_std', prior_range=[1.5, 2, 2.5], prior_default_dict=prior_default_dict)
# varying sigma_mean
varying_priors(data_object=sensitivity, prior_name='sigma_mean', prior_range=[0.3, 0.4, 0.5], prior_default_dict=prior_default_dict)
# varying sigma_std
varying_priors(data_object=sensitivity, prior_name='sigma_std', prior_range=[0.3, 0.4, 0.5], prior_default_dict=prior_default_dict)
# varying sigma_lower_limit
varying_priors(data_object=sensitivity, prior_name='sigma_lower_limit', prior_range=[0.1, 0.2, 0.3], prior_default_dict=prior_default_dict)



Define plotting function

In [42]:
# plotting function for prior effects, S5-S9
def plot_prior_effect(prior_name, prior_range):
    df = pd.read_csv(os.path.join(save_output_path, f'varying_{prior_name}.csv'))
    fig, ax = plt.subplots(3, 3, figsize=(11, 6), sharex=True, sharey=False)
    sns.set_style('white')
    sns.set_context('paper')
    axes = [ax[0][0], ax[0][1], ax[0][2],
            ax[1][0], ax[1][1], ax[1][2],
            ax[2][0], ax[2][1], ax[2][2],]
    for i, compound in enumerate(selected_compounds):
        this_df = df[df['compound_name'] == compound]
        this_ax = axes[i]
        colors = sns.color_palette("icefire", 3)
        avg_probabilities = {}
        for c, prior_value in enumerate(prior_range):
            by_value_df = this_df[this_df[prior_name] == prior_value]
            color = colors[c]
            for seed in [1,2,3]:
                row = by_value_df[by_value_df['random_state'] == seed]
                assert not row.empty, f"Row is empty, data not found for {compound}, {prior_value}, {seed} \n {this_df}"
                this_ax.set_xlim(-2.5, 4.5)
                this_ax.axvspan(-1, 3, color='lightgrey', alpha=0.1, lw=0, label='LOQ range')
                x_range = np.arange(-2.5, 4.5, 0.001)
                this_ax.plot(x_range, norm.pdf(x_range, row['logDT50_mean'], row['logDT50_uncert']),
                # sns.kdeplot(np.random.normal(loc=row['logDT50_mean'], scale=row['logDT50_uncert'], size=10000), ax=this_ax, 
                            color=color, alpha=.5, linewidth=1,
                            label=prior_value)
            avg_probabilities[prior_value] = {'p(nP)': by_value_df['p(nP)'].mean(),
                                              'p(P)': by_value_df['p(P)'].mean(),
                                              'p(vP)': by_value_df['p(vP)'].mean(),
                                              'color': color,
                                              }
            
        _, max_y = this_ax.get_ylim()
        min_x, _ = this_ax.get_xlim()    
            
        this_ax.text(-2.4, max_y*0.95, "{}, n={}".format(compound, row['DT50_count'].values[0]))
        
        start_y = max_y*0.85
        for key, value in avg_probabilities.items():
            if i in [4,7]:
                this_ax.text(3, start_y, f"p(P) = {np.round(value['p(vP)'],2)}", color=value['color'])
            else:
                this_ax.text(-2.4, start_y, f"p(P) = {np.round(value['p(vP)'],2)}", color=value['color'])
            start_y -= max_y*0.1
        
        this_ax.plot([np.log10(120), np.log10(120)], [0, max_y], label='P threshold', color='black', ls='--')
        this_ax.plot([np.log10(180), np.log10(180)], [0, max_y], label='vP threshold', color='black', ls=':')
        if i >= 6:
            this_ax.set_xlabel('log(DT50)')
    
    plt.tight_layout()
    plt.savefig(os.path.join(save_output_path, f'varying_{prior_name}.pdf'))
    plt.close()

Plot prior effects for each prior parameter

In [41]:
plot_prior_effect('mu_mean', [0.5, 1, 1.5])
plot_prior_effect('mu_std', [1.5, 2, 2.5])
plot_prior_effect('sigma_mean', [0.3, 0.4, 0.5])
plot_prior_effect('sigma_std', [0.3, 0.4, 0.5])
plot_prior_effect('sigma_lower_limit', [0.1, 0.2, 0.3])


### S4.1 Model cross-validation
Define paths to output files of 5-fold CV

In [79]:
# define output paths
directory_path = os.path.join("..", "..", "pepper_data", "modeling", "scores", "nested_cross_validation_screening", "default_curation")
                              
predicted_file_path_distance_cv_gpr = os.path.join(directory_path, "GPR", "predicted_target_variable_test_padel_None_uncertainty_paper_soil_soil_all_data.tsv")
predicted_file_path_distance_cv_rf = os.path.join(directory_path, "RF","predicted_target_variable_test_all_None_uncertainty_paper_soil_soil_all_data.tsv")  
all_predicted_files = {
    "GPR": [
            "predicted_target_variable_test_all_None_uncertainty_paper_soil_soil_all_data.tsv",
            "predicted_target_variable_test_all_pca_uncertainty_paper_soil_soil_all_data.tsv",
            "predicted_target_variable_test_padel_None_uncertainty_paper_soil_soil_all_data.tsv",
            "predicted_target_variable_test_padel_pca_uncertainty_paper_soil_soil_all_data.tsv",
            "predicted_target_variable_test_rdkitfps_None_uncertainty_paper_soil_soil_all_data.tsv",
            "predicted_target_variable_test_rdkitfps_pca_uncertainty_paper_soil_soil_all_data.tsv",
            "predicted_target_variable_test_rdkitdesc_None_uncertainty_paper_soil_soil_all_data.tsv",
            "predicted_target_variable_test_rdkitdesc_pca_uncertainty_paper_soil_soil_all_data.tsv",
            "predicted_target_variable_test_avalonfps_None_uncertainty_paper_soil_soil_all_data.tsv",
            "predicted_target_variable_test_avalonfps_pca_uncertainty_paper_soil_soil_all_data.tsv",
            "predicted_target_variable_test_maccs_None_uncertainty_paper_soil_soil_all_data.tsv",
            "predicted_target_variable_test_maccs_pca_uncertainty_paper_soil_soil_all_data.tsv",
            "predicted_target_variable_test_eptrig_None_uncertainty_paper_soil_soil_all_data.tsv",
            "predicted_target_variable_test_eptrig_pca_uncertainty_paper_soil_soil_all_data.tsv"
        ]
,   
    "RF": [
            "predicted_target_variable_test_all_None_uncertainty_paper_soil_soil_all_data.tsv",
            "predicted_target_variable_test_padel_None_uncertainty_paper_soil_soil_all_data.tsv",
            "predicted_target_variable_test_rdkitfps_None_uncertainty_paper_soil_soil_all_data.tsv",
            "predicted_target_variable_test_rdkitdesc_None_uncertainty_paper_soil_soil_all_data.tsv",
            "predicted_target_variable_test_avalonfps_None_uncertainty_paper_soil_soil_all_data.tsv",
            "predicted_target_variable_test_maccs_None_uncertainty_paper_soil_soil_all_data.tsv",
            "predicted_target_variable_test_eptrig_None_uncertainty_paper_soil_soil_all_data.tsv",
            ]        
}

Define functions

In [73]:
# define function to plot uncertainty distributions
def plot_uncertainty_distribution(path, name='GPR'):
    predicted = pd.read_csv(path, sep="\t")

    sns.set(rc={"figure.figsize": (5, 5)})
    sns.set_theme(style="whitegrid")
    sns.set_style("ticks")
    sns.set_context(context)
    output_file_path = os.path.join(directory_path,
                            'uncertainty_distribution_{}_{}_{}_{}.pdf'.format(
                                'cv',
                                name,
                                'padel',
                                'soil',
                            ))

    # if name == 'GPR':
    #     predicted['predicted_score'] = np.sqrt(predicted['predicted_score']**2 - predicted['experimental_std']**2)
    ax = sns.histplot(x='predicted_score', data=predicted, hue='setup_name', kde=True, stat="density", common_norm=False)

    print("Save ")
    plt.title('Uncertainty Distribution - {}'.format(name))
    plt.savefig(output_file_path)
    plt.show()
    plt.close()

    plt.figure(figsize=(8, 6))
    ax = sns.scatterplot(y='nearest_distance_1', x='predicted_score', data=predicted, hue='setup_name')
    plt.xlim(0, 1.8)
    plt.ylim(0, 1)

    plt.title('Uncertainty vs Nearest Distance - {}'.format(name))
    plt.show()
    plt.close()

    # extract arrays
    pred_sigma = predicted['predicted_score'].values
    true_sigma = predicted['experimental_std'].values

    # define bins
    n_bins = np.sqrt(len(predicted)).astype(int)
    bins = np.linspace(pred_sigma.min(), pred_sigma.max(), n_bins + 1)
    bin_centers = 0.5 * (bins[:-1] + bins[1:])

    # compute empirical std in each bin
    empirical_std = []
    for i in range(n_bins):
        mask = (pred_sigma >= bins[i]) & (pred_sigma < bins[i+1])
        if mask.sum() > 1:
            empirical_std.append(true_sigma[mask].mean())
        else:
            empirical_std.append(np.nan)

    # --- Plot ---
    plt.figure(figsize=(8, 6))

    # scatter for calibration curve
    plt.scatter(bin_centers, empirical_std, s=80, label='Empirical std (binned)')

    # ideal diagonal
    

    plt.xlabel("Predicted std")
    plt.ylabel("Empirical std")
    plt.title("Standard Deviation Calibration (Binned)")
    plt.xlim(0.4, 1.0)
    plt.ylim(0.0, 0.7)
    plt.legend()
    plt.show()
    plt.close()

    # correlation of predicted vs empirical std

    spearman_r = spearmanr(pred_sigma, true_sigma)
    pearson_r = pearsonr(pred_sigma, true_sigma)[0]
    print(f"{name} - Spearman correlation: {spearman_r.correlation:.4f} (p={spearman_r.pvalue:.4e})")
    print(f"{name} - Pearson correlation: {pearson_r:.4f}")
    plt.figure(figsize=(8, 6))
    plt.scatter(pred_sigma, true_sigma, alpha=0.3)
    plt.xlabel("Predicted std")
    plt.ylabel("Empirical std")
    plt.title("Standard Deviation Correlation")
    

    plt.show()
    plt.close()

    # error-based calibration plot

    # add aleatoric uncertainty to predicted uncertainty 

    # predicted['predicted_score'] = np.sqrt(predicted['predicted_score']**2 + predicted['experimental_std']**2)

    batches = np.sqrt(len(predicted))  # number of batches

    # sort the predictions by predicted uncertainty
    df_sorted = predicted.sort_values(by='predicted_score').reset_index(drop=True)
    batch_size = int(np.ceil(len(df_sorted) / batches))


    empirical_rmse = []
    empirical_rmu = []
    empirical_min_distance = []

    for i in range(int(batches)):
        start_index = i * batch_size
        end_index = min((i + 1) * batch_size, len(df_sorted))
        batch = df_sorted.iloc[start_index:end_index]

        rmse = root_mean_squared_error(batch['experimental'], batch['predicted'])
        rmu = np.sqrt(np.mean(batch['predicted_score']**2))
        min_distance = np.mean(batch['nearest_distance_1'])

        empirical_rmse.append(rmse)
        empirical_rmu.append(rmu)
        empirical_min_distance.append(min_distance)

    # calculate ENCE

    ence = np.mean(np.abs(np.array(empirical_rmse) - np.array(empirical_rmu)) / np.array(empirical_rmu))
    
    # plot error-based calibration curve
    sns.set(rc={"figure.figsize": (5, 5)})
    sns.set_theme(style="whitegrid")
    sns.set_style("ticks")
    sns.set_context(context)
    plt.scatter(empirical_rmu, empirical_rmse, marker='o', label='Empirical')
    plt.plot([0, max(empirical_rmu + empirical_rmse)], [0, max(empirical_rmu + empirical_rmse)], ls='--', color='black', label='Ideal')
    plt.xlabel('Empirical RMU')
    plt.ylabel('Empirical RMSE')
    plt.title('Error-based Calibration Plot')
    plt.legend()
    textstr = f'ENCE = {ence:.3f}\nn={len(predicted)}, batches={int(batches)}'
    plt.text(0.95, 0.05, textstr, fontsize=10)
    plt.show()
    plt.close()


    # plot rmse vs predicted uncertainty
    plt.figure(figsize=(8, 6))
    predicted['absolute_error'] = np.abs(predicted['predicted'] - predicted['experimental'])
    predicted['squared_error'] = (predicted['predicted'] - predicted['experimental'])**2    
    plt.scatter(predicted['predicted_score'], predicted['squared_error'], alpha=0.3)
    plt.xlabel("Predicted std")
    plt.ylabel("Absolute Error")
    plt.title("Absolute Error vs Predicted Uncertainty")
    plt.show()
    plt.close()


In [76]:
# define functions for calibration plots
def plot_calibration(path, name):
    file_name = path.split("/")[-1].replace(".tsv", "")
    file_name_parts = file_name.split("_")
    feature_name = file_name_parts[4]
    feature_reduction = file_name_parts[5]

    sns.set(rc={"figure.figsize": (5, 5)})
    sns.set_theme(style="whitegrid")
    sns.set_style("ticks")
    sns.set_context(context)
    text_fontsize = 7
    
    # ---------
    # read data
    # ---------
    predicted = pd.read_csv(path, sep="\t")

    predicted['fold'] = predicted['setup_name']

    print("\nUncertainty distribution range:", predicted['predicted_score'].min(), "to", predicted['predicted_score'].max())
    n_bins = np.sqrt(len(predicted)) 

    # ------------
    # calibrations
    # ------------
    # confidence based calibration 
    nominal_levels = [0.05, 0.1, 0.15, 0.2, 0.25, 0.3, 0.35, 0.4, 0.45, 0.5, 0.55, 0.6, 0.65, 0.7, 0.75, 0.8, 0.85, 0.9, 0.95, 1]

    # calculate empirical levels
    empirical_levels = []
    for alpha in nominal_levels:
        z = norm.ppf((1 + alpha) / 2)
        lower = predicted['predicted'] - z * predicted['predicted_score']
        upper = predicted['predicted'] + z * predicted['predicted_score']
        count_within_interval = np.sum((predicted['experimental'] >= lower) & (predicted['experimental'] <= upper))
        empirical_level = count_within_interval / len(predicted)
        empirical_levels.append(empirical_level)

    ece = np.mean(np.abs(np.array(nominal_levels) - np.array(empirical_levels)))

    # error & distance based calibration plot

    # Sort for uncertainty 
    df_sorted = predicted.sort_values('predicted_score').reset_index(drop=True)
    batch_size= int(np.ceil(len(df_sorted) / n_bins))

    empirical_rmse = []
    empirical_rmu = []
    empirical_min_distance = []

    for i in range(int(n_bins)):
        start_index = i * batch_size
        end_index = min((i + 1) * batch_size, len(df_sorted))
        batch = df_sorted.iloc[start_index:end_index]

        rmse = root_mean_squared_error(batch['experimental'], batch['predicted'])
        rmu = np.sqrt(np.mean(batch['predicted_score']**2))
        min_distance = np.mean(batch['nearest_distance_1'])

        empirical_rmse.append(rmse)
        empirical_rmu.append(rmu)
        empirical_min_distance.append(min_distance)

    ence = np.mean(np.abs(np.array(empirical_rmse) - np.array(empirical_rmu)) / np.array(empirical_rmu))

    sigmas = predicted['predicted_score']
    mu_sigma = np.mean(sigmas)
    variance = np.sum((sigmas - mu_sigma)**2) /(len(predicted) -1)
    c_v = np.sqrt(variance) / mu_sigma

    spearmans_dist = spearmanr(empirical_rmu, empirical_min_distance)

    # error and distance based calibration per fold 
    folds_df = {"empirical_rmse_folds": [], 
                "empirical_rmu_folds": [], 
                "empirical_min_distance_folds": [],
                "fold": []
                }

    for fold in predicted['fold'].unique():
        df_fold = predicted[predicted['fold'] == fold]
        df_sorted_fold = df_fold.sort_values('predicted_score').reset_index(drop=True)
        batch_size_fold = int(np.ceil(len(df_sorted_fold) / n_bins))

        for i in range(int(n_bins)):
            start_index = i * batch_size_fold
            end_index = min((i + 1) * batch_size_fold, len(df_sorted_fold))
            batch = df_sorted_fold.iloc[start_index:end_index]

            rmse = root_mean_squared_error(batch['experimental'], batch['predicted'])
            rmu = np.sqrt(np.mean(batch['predicted_score']**2))
            min_distance = np.mean(batch['nearest_distance_1'])
        
            folds_df["empirical_rmse_folds"].append(rmse)
            folds_df["empirical_rmu_folds"].append(rmu)
            folds_df["empirical_min_distance_folds"].append(min_distance)
            folds_df["fold"].append(fold)

    ence_folds = {}
    for fold in predicted['fold'].unique():
        ence_folds[fold] = np.mean(np.abs(np.array(folds_df["empirical_rmse_folds"]) - np.array(folds_df["empirical_rmu_folds"])) / np.array(folds_df["empirical_rmu_folds"]))
    
    spearmans_dist_folds = {}
    for fold in predicted['fold'].unique():
        spearmans_dist_folds[fold] = spearmanr(folds_df["empirical_rmu_folds"], folds_df["empirical_min_distance_folds"])

    #-----------
    # fold plots
    #-----------
    fig, axs = plt.subplots(1,2, figsize=(10, 5))
    (ax_err_fold, ax_dist_fold) = axs.flatten()
    
    df_folds = pd.DataFrame(folds_df)

    # error based calibration fold plot
    sns.scatterplot(x='empirical_rmu_folds', y='empirical_rmse_folds', ax=ax_err_fold, marker='o', hue='fold', data=df_folds)
    ax_err_fold.plot([0, max(df_folds['empirical_rmu_folds'] + df_folds['empirical_rmse_folds'])], [0, max(df_folds['empirical_rmu_folds'] + df_folds['empirical_rmse_folds'])], ls='--', color='black', label='Ideal')
    ax_err_fold.set_xlabel('Empirical RMU')
    ax_err_fold.set_ylabel('Empirical RMSE')
    ax_err_fold.legend()
    ax_err_fold.set_xlim(df_folds['empirical_rmu_folds'].min(), df_folds['empirical_rmu_folds'].max() * 1.1)
    # textstr = f'ENCE = {ence:.3f}\nn={len(predicted)}, bins={int(n_bins)}'
    # ax_err_fold.text(0.95, 0.05, textstr, fontsize=text_fontsize, verticalalignment='bottom', horizontalalignment='right', transform=ax_err_fold.transAxes)

    # distance vs uncertainty fold plot
    sns.scatterplot(x='empirical_rmu_folds', y='empirical_min_distance_folds', alpha=0.5, ax=ax_dist_fold, hue='fold', data=df_folds)
    ax_dist_fold.set_xlabel('Empirical RMU')
    ax_dist_fold.set_ylabel('Nearest Distance')
    # textstr = f'Spearman r = {spearmans_dist.correlation:.2f}\nn={len(predicted)}, batches={int(n_bins)}'
    # ax_dist.text(0.95, 0.05, textstr,
    #     fontsize=text_fontsize, verticalalignment='bottom', horizontalalignment='right',
    #     transform=ax_dist.transAxes)

    plt.tight_layout()
    save_path = os.path.join(directory_path,
                            'calibration_plots_folds_{}_{}_{}.pdf'.format(name, feature_name, feature_reduction))
    print("\nSaving figure to: ", save_path)
    plt.savefig(
        save_path,
        dpi=600
    )
    plt.close()

    # --------
    # subplots
    # --------
    fig, axs = plt.subplots(3,2, figsize=(9, 12))
    (ax_parity, ax_unc, ax_conf, ax_err, ax_dist, ax_empty) = axs.flatten()

    # parity plot
    sns.scatterplot(x='experimental', y='predicted', data=predicted, hue='fold', alpha=0.5, ax=ax_parity)

    # Add +/-1 log unit lines
    min_val = min(predicted['experimental'].min(), predicted['predicted'].min()) * 0.9
    max_val = max(predicted['experimental'].max(), predicted['predicted'].max()) * 1.1

    ax_parity.plot([min_val, max_val], [min_val, max_val], ls='-', color='black')
    ax_parity.plot([min_val + 1, max_val], [min_val, max_val - 1], ls='--', color='black')
    ax_parity.plot([min_val, max_val - 1], [min_val + 1, max_val], ls='--', color='black')
    ax_parity.set(xlim=[min_val, max_val], ylim=[min_val, max_val])
    ax_parity.legend_.set_title(None)
    # compute metrics per fold
    folds = predicted['fold'].unique()
    textstr = ''
    for fold in folds:
        df_fold = predicted[predicted['fold'] == fold]
        r2 = r2_score(df_fold['experimental'], df_fold['predicted'])
        rmse = root_mean_squared_error(df_fold['experimental'], df_fold['predicted'])
        textstr += f'Fold {fold[-1]}: R2 = {r2:.2f}, RMSE = {rmse:.2f}, n = {len(df_fold)}\n'

    # average over all folds
    r2_all = r2_score(predicted['experimental'], predicted['predicted'])
    rmse_all = root_mean_squared_error(predicted['experimental'], predicted['predicted'])

    # add a line for average
    textstr += '-----------------------------------------------------\n'
    textstr += f'Avg. :   R2 = {r2_all:.2f}, RMSE = {rmse_all:.2f}, n = {len(predicted)}\n'

    # add text box to plot
    ax_parity.text(0.95, 0.05, textstr,
        fontsize=text_fontsize, verticalalignment='bottom', horizontalalignment='right',
        transform=ax_parity.transAxes)

    # uncertainty distribution plots
    sns.histplot(x='predicted_score', data=predicted,hue='fold', kde=True, stat="count", common_norm=False, ax=ax_unc)
    ax_unc.set_xlabel('Predicted Uncertainty')
    ax_unc.set_ylabel('Count')

    # calibration plot
    sns.scatterplot(x=nominal_levels, y=empirical_levels, ax=ax_conf, marker='o')
    ax_conf.plot([0, 1], [0, 1], ls='--', color='black', label='Ideal')
    ax_conf.set_xlabel('Nominal Confidence Level')
    ax_conf.set_ylabel('Empirical Confidence Level')
    ax_conf.legend()
    textstr = f'ECE = {ece:.3f}\nn={len(predicted)}'
    ax_conf.text(0.95, 0.05, textstr, fontsize=text_fontsize, verticalalignment='bottom', horizontalalignment='right', transform=ax_conf.transAxes)

    # error based calibration plot
    sns.scatterplot(x=empirical_rmu, y=empirical_rmse, ax=ax_err, marker='o')
    ax_err.plot([0, max(empirical_rmu + empirical_rmse)], [0, max(empirical_rmu + empirical_rmse)], ls='--', color='black', label='Ideal')
    ax_err.set_xlabel('Empirical RMU')
    ax_err.set_ylabel('Empirical RMSE')
    ax_err.legend()
    textstr = f'ENCE = {ence:.3f}\nc_v = {c_v:.3f}, n={len(predicted)}, bins={int(n_bins)}\nspearman r = {spearmanr(empirical_rmu, empirical_rmse).correlation:.2f}\npearson r = {pearsonr(empirical_rmu, empirical_rmse)[0]:.2f}'
    ax_err.text(0.95, 0.05, textstr, fontsize=text_fontsize, verticalalignment='bottom', horizontalalignment='right', transform=ax_err.transAxes)

    # distance vs uncertainty plot
    sns.scatterplot(x=empirical_rmu, y=empirical_min_distance, alpha=0.5, ax=ax_dist)
    ax_dist.set_xlabel('Empirical RMU')
    ax_dist.set_ylabel('Nearest Distance')
    textstr = f'Spearman r = {spearmans_dist.correlation:.2f}\npearson r = {pearsonr(empirical_rmu, empirical_min_distance)[0]:.2f}\nn={len(predicted)}, batches={int(n_bins)}'
    ax_dist.text(0.95, 0.05, textstr,
        fontsize=text_fontsize, verticalalignment='bottom', horizontalalignment='right',
        transform=ax_dist.transAxes)
    
    # remove empty plot 
    # ax_empty.axis('off')
    
    # add scatterplot for "good" cv predictions
    if name == 'GPR':
        threshold = 0.5
    else:
        threshold = 0.5
        
    good_predictions_only = predicted[predicted['predicted_score'] <= threshold] # good predictions only
    sns.scatterplot(x='experimental', y='predicted', data=good_predictions_only, hue='fold', alpha=0.5, ax=ax_empty)
    ax_empty.plot([min_val, max_val], [min_val, max_val], ls='-', color='black')
    ax_empty.plot([min_val + 1, max_val], [min_val, max_val - 1], ls='--', color='black')
    ax_empty.plot([min_val, max_val - 1], [min_val + 1, max_val], ls='--', color='black')
    ax_empty.set(xlim=[min_val, max_val], ylim=[min_val, max_val])
    ax_empty.legend_.set_title(None)
    # compute metrics per fold
    textstr = 'Prediction score <= 0.5:\n'
    textstr += '-----------------------------------------------------\n'

    for fold in folds:
        df_fold = good_predictions_only[good_predictions_only['fold'] == fold]
        if df_fold.empty:
            r2 = rmse = np.nan
        else:
            r2 = r2_score(df_fold['experimental'], df_fold['predicted'])
            rmse = root_mean_squared_error(df_fold['experimental'], df_fold['predicted'])
        textstr += f'Fold {fold[-1]}: R2 = {r2:.2f}, RMSE = {rmse:.2f}, n = {len(df_fold)}\n'

    # average over all folds
    r2_all = r2_score(good_predictions_only['experimental'], good_predictions_only['predicted'])
    rmse_all = root_mean_squared_error(good_predictions_only['experimental'], good_predictions_only['predicted'])

    # add a line for average
    textstr += '-----------------------------------------------------\n'
    textstr += f'Avg. :   R2 = {r2_all:.2f}, RMSE = {rmse_all:.2f}, n = {len(good_predictions_only)}\n'

    # add text box to plot
    ax_empty.text(0.95, 0.05, textstr,
        fontsize=text_fontsize, verticalalignment='bottom', horizontalalignment='right',
        transform=ax_empty.transAxes)
    
    fig.tight_layout()
    save_path = os.path.join(directory_path,
                            'calibration_plots_{}_{}_{}.pdf'.format(name, feature_name, feature_reduction))
    print("\nSaving figure to: ", save_path)
    plt.savefig(
        save_path,
        dpi=600
    )
    plt.close()

    # summarize all metrics in a tsv file
    summary_dict = {
        'feature_name': feature_name,
        'feature_reduction': feature_reduction,
        '$R^2$': r2_all,
        'RMSE': rmse_all,
        'ece': ece,
        'ence': ence,
        'c_v': c_v,
        'spearman_r_dist': spearmans_dist.correlation,
        'pearson_r_dist': pearsonr(empirical_rmu, empirical_min_distance)[0],
    }
    print(summary_dict)
    

In [86]:
# define function for training uncertainty
def plot_train_uncertainty(path, name='GPR',ax=None):
    predicted = pd.read_csv(path, sep="\t")

    file_name = path.split("/")[-1].replace(".tsv", "")
    file_name_parts = file_name.split("_")
    feature_name = file_name_parts[4]
    feature_reduction = file_name_parts[5]

    sns.set(rc={"figure.figsize": (5, 5)})
    sns.set_theme(style="whitegrid")
    sns.set_style("ticks")
    sns.scatterplot(x='predicted_score', y='experimental_std', data=predicted, s=8, alpha=0.5, ax=ax)

    ax.set_xlabel("Predicted Std")
    if name == 'RF':
        ax.set_ylabel("Experimental Std")
    else:
        ax.set_ylabel("")
        ax.set_yticklabels([])

Produce training set fitted uncertainty plots: Figure S12

In [ ]:
# plot train_experimental_vs_fitted_std.pdf
predicted_file_path_distance_cv_gpr_train = os.path.join(directory_path, "GPR", "predicted_target_variable_train_padel_None_uncertainty_paper_soil_soil_all_data.tsv")
predicted_file_path_distance_cv_rf_train = os.path.join(directory_path, "RF", "predicted_target_variable_train_all_None_uncertainty_paper_soil_soil_all_data.tsv")

fig, ax = plt.subplots(1,2, figsize=(10, 4), )
plot_train_uncertainty(predicted_file_path_distance_cv_rf_train, name='RF', ax=ax[0])
plot_train_uncertainty(predicted_file_path_distance_cv_gpr_train, name='GPR', ax=ax[1])

output_file_path = os.path.join(directory_path,
                    'train_experimental_vs_fitted_std.pdf')
print("Save to:", output_file_path)
plt.tight_layout()
plt.savefig(output_file_path)
plt.show()
plt.close()

Figures S10 and S11: Performance and uncertainty analysis, 6 panels

In [ ]:
# calibration plots (6 panels)
for rf_file in all_predicted_files['RF']:
    rf_path = os.path.join(directory_path, "RF", rf_file)
    plot_calibration(rf_path, name='RF')

for gpr_file in all_predicted_files['GPR']:
    gpr_path = os.path.join(directory_path, "GPR", gpr_file)
    plot_calibration(gpr_path, name='GPR')


Table S5: Model results accross all configurations
Model performance plots can only be created, when calling soil_modeling.nested_cross_validation_screening in soil_paper.py

In [ ]:
# Model performance plots
file_path_uncertainty_paper_soil = os.path.join(directory_path, "RF","test_scores_all_uncertainty_paper_soil_soil_all_data.tsv")

df = pd.read_csv(file_path_uncertainty_paper_soil, sep="\t")

df["feature_reduction"] = df["feature_reduction"].fillna("None")
df["model_label"] = df["regressor"] + " | " + df["feature_reduction"]

# Choose your metrics
metrics = ["R2", "MSE", "RMSE", "MAE"]

sns.set(rc={"figure.figsize": (5, 5)})
sns.set_theme(style="whitegrid")
sns.set_style("ticks")
sns.set_context(context)

for metric in metrics:
    plt.figure(figsize=(12, 6))
    ax = sns.boxplot(
        data=df,
        x="descriptors",
        y=metric,
        hue="model_label",
        palette="Paired",
        width=0.6,  # narrower boxes = more space between groups
        showfliers=False
    )
    sns.stripplot(
        data=df,
        x="descriptors",
        y=metric,
        hue="model_label",
        dodge=True,
        palette='dark:black',
        size=3,
        alpha=0.5, legend=False
    )
    plt.title(f"{metric.upper()} by Descriptor")
    plt.xlabel("Descriptor")
    plt.ylabel(metric.upper())
    plt.xticks(rotation=45)
    plt.legend(
        bbox_to_anchor=(1.05, 1),
        loc="upper left",
        title="Model | Feature Reduction"
    )
    plt.tight_layout()
    plt.savefig(os.path.join(save_output_path, f"model_performance_{metric}.pdf"), dpi=600)
    plt.show()

In [65]:
# Model performance summary
# Compute mean across folds for each combination
summary = (
    df.groupby(["regressor", "feature_reduction", "descriptors"])
      .agg({
          "R2": "mean",
          "MSE": "mean",
          "RMSE": "mean",
          "MAE": "mean"
      })
      .reset_index()
)

# Optional: add overall ranking by R² (best first)
summary["rank_r2"] = summary["R2"].rank(ascending=False, method="dense").astype(int)
summary = summary.sort_values("R2", ascending=False)

summary = summary.round(2)

# Display the summary nicely
summary.to_csv(os.path.join(save_output_path, "model_performance_summary.tsv"), sep="\t", index=False)

In [66]:
# Compute Fold metrics
def compute_fold_metrics(path, name):
    """
    predicted: DataFrame with columns
        'experimental', 'predicted', 'predicted_score', 'nearest_distance_1', 'fold'
    returns: DataFrame with metrics per fold
    """

    predicted = pd.read_csv(path, sep="\t")

    nominal_levels = [0.1, 0.15, 0.2, 0.25, 0.3, 0.35, 0.4, 0.45, 0.5,
                      0.55, 0.6, 0.65, 0.7, 0.75, 0.8, 0.85, 0.9, 0.95, 1.0]

    cv_metrics = []
    # ---------- point metrics ----------
    r2 = r2_score(predicted['experimental'], predicted['predicted'])
    rmse = root_mean_squared_error(predicted['experimental'], predicted['predicted'])

    # ---------- ECE ----------
    empirical_levels = []
    for alpha in nominal_levels:
        z = norm.ppf((1 + alpha) / 2)
        lower = predicted['predicted'] - z * predicted['predicted_score']
        upper = predicted['predicted'] + z * predicted['predicted_score']
        count_within_interval = np.sum(
            (predicted['experimental'] >= lower) & 
            (predicted['experimental'] <= upper)
        )
        empirical_level = count_within_interval / len(predicted)
        empirical_levels.append(empirical_level)

    ece = np.mean(np.abs(np.array(nominal_levels) - np.array(empirical_levels)))

    # ---------- ENCE & Spearman ----------
    # choose bins per fold (same rule as your global one)
    n_bins = int(np.sqrt(len(predicted)))
    if n_bins < 2:  # guard for very small folds
        n_bins = 2

    df_sorted = predicted.sort_values('predicted_score').reset_index(drop=True)
    batch_size = int(np.ceil(len(df_sorted) / n_bins))

    empirical_rmse = []
    empirical_rmu = []
    empirical_min_distance = []

    for i in range(n_bins):
        start_index = i * batch_size
        end_index = min((i + 1) * batch_size, len(df_sorted))
        batch = df_sorted.iloc[start_index:end_index]
        if len(batch) == 0:
            continue

        rmse_bin = root_mean_squared_error(batch['experimental'], batch['predicted'])
        rmu_bin = np.sqrt(np.mean(batch['predicted_score'] ** 2))
        min_distance = np.mean(batch['nearest_distance_1'])

        empirical_rmse.append(rmse_bin)
        empirical_rmu.append(rmu_bin)
        empirical_min_distance.append(min_distance)

    empirical_rmse = np.array(empirical_rmse)
    empirical_rmu = np.array(empirical_rmu)

    ence = np.mean(np.abs(empirical_rmse - empirical_rmu) / empirical_rmu)

    spearmans_dist = spearmanr(empirical_rmu, empirical_min_distance).correlation
    pearson_r_dist = pearsonr(empirical_rmu, empirical_min_distance)[0]

    # ---------- store ----------
    cv_metrics.append({
        'R2': round(r2, 2),
        'RMSE': round(rmse, 2),
        'ECE': round(ece * 100, 1),  # convert to percentage
        'ENCE': round(ence * 100, 1),
        'Spearman_dist': round(spearmans_dist, 2),
        'Pearson_dist': round(pearson_r_dist, 2),
    })

    return pd.DataFrame(cv_metrics)

In [67]:
# Table S5
model_results = {}
for rf_file in all_predicted_files['RF']:
    rf_path = os.path.join(directory_path, "RF", rf_file)
    feature_name = rf_file.split("_")[4]
    feature_reduction = rf_file.split("_")[5]
    df_metrics = compute_fold_metrics(rf_path, name='RF')
    model_results[f'RF_{feature_name}_{feature_reduction}'] = df_metrics

for gpr_file in all_predicted_files['GPR']:
    gpr_path = os.path.join(directory_path, "GPR", gpr_file)
    feature_name = gpr_file.split("_")[4]
    feature_reduction = gpr_file.split("_")[5]
    df_metrics = compute_fold_metrics(gpr_path, name='GPR')
    model_results[f'GPR_{feature_name}_{feature_reduction}'] = df_metrics



combined_rows = []

for model_name, df in model_results.items():
    df = df.copy()
    split_name = model_name.split("_")
    df["model"] = split_name[0]      # add model type as a
    df["features"] = split_name[1]               # add model name as a column
    df["reduction"] = split_name[2]               # add model name as a column
    combined_rows.append(df)

combined = pd.concat(combined_rows, ignore_index=True)

output_path = os.path.join(directory_path, "all_model_metrics.csv")
combined.to_csv(output_path, index=False)
combined.to_latex(output_path.replace(".csv", ".tex"), index=False, float_format="%.2f")
print("Saved:", output_path)

Saved: ../../pepper_data/modeling/scores/nested_cross_validation_screening/default_curation/all_model_metrics.csv


In [80]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import norm

def plot_confidence_calibration(path, name='GPR', for_folds=False):
    predicted = pd.read_csv(path, sep="\t")
    
    # define nominal levels
    nominal_levels = [0.1, 0.15, 0.2, 0.25, 0.3, 0.35, 0.4, 0.45,
                      0.5, 0.55, 0.6, 0.65, 0.7, 0.75, 0.8, 0.85,
                      0.9, 0.95, 1.0]

    def compute_empirical_levels(fold_data):
        empirical_levels = []
        for alpha in nominal_levels:
            z = norm.ppf((1 + alpha) / 2)
            lower = fold_data['predicted'] - z * fold_data['predicted_score']
            upper = fold_data['predicted'] + z * fold_data['predicted_score']
            count_within_interval = np.sum(
                (fold_data['experimental'] >= lower) &
                (fold_data['experimental'] <= upper)
            )
            empirical_level = count_within_interval / len(fold_data)
            empirical_levels.append(empirical_level)
        ece = np.mean(np.abs(np.array(empirical_levels) - np.array(nominal_levels)))
        return empirical_levels, ece

    # ------------------------------------------------------------------
    # Plot setup
    # ------------------------------------------------------------------
    sns.set(rc={"figure.figsize": (5, 5)})
    sns.set_theme(style="whitegrid")
    sns.set_style("ticks")
    sns.set_context(context)

    if for_folds:
        # Build a long dataframe: one row per (fold, nominal_level)
        rows = []
        ece_folds = []

        for fold in predicted['setup_name'].unique():
            fold_data = predicted[predicted['setup_name'] == fold]
            empirical_levels, ece = compute_empirical_levels(fold_data)
            ece_folds.append((fold, ece))

            for nl, el in zip(nominal_levels, empirical_levels):
                rows.append({
                    'Fold': fold,
                    'Nominal': nl,
                    'Empirical': el
                })

        df = pd.DataFrame(rows)

        # Plot calibration per fold
        ax = sns.lineplot(
            data=df,
            x='Nominal',
            y='Empirical',
            hue='Fold',
            marker='o'
        )

        # Add ideal line
        ax.plot([0.0, 1], [0.0, 1], ls='--', color='black', label='Ideal')

        ax.set_xlabel('Nominal confidence level')
        ax.set_ylabel('Empirical confidence level')
        ax.set_xlim(-0.03, 1.03)
        ax.set_ylim(-0.03, 1.03)
        ax.set_title(f'Calibration: {name}')

        # Create an ECE text with per-fold values
        ece_str = '\n'.join(f'{fold}: {ece:.3f}' for fold, ece in ece_folds)
        textstr = f'ECE per fold:\n{ece_str}'
        ax.text(0.8, 0.05, textstr, transform=ax.transAxes,
                fontsize=9, va='bottom', ha='left')

        # Make sure legend includes the ideal line
        handles, labels = ax.get_legend_handles_labels()
        if 'Ideal' not in labels:
            handles.append(plt.Line2D([0], [0], ls='--', color='black'))
            labels.append('Ideal')
        ax.legend(handles, labels)

    else:
        # Single calibration using all data
        empirical_levels, ece = compute_empirical_levels(predicted)
        df = pd.DataFrame({
            'Nominal': nominal_levels,
            'Empirical': empirical_levels
        })

        ax = sns.lineplot(
            data=df,
            x='Nominal',
            y='Empirical',
            marker='o',
            label='Empirical'
        )
        ax.plot([0.01, 1], [0.01, 1], ls='--', color='black', label='Ideal')
        ax.set_xlabel('Nominal confidence level')
        ax.set_ylabel('Empirical confidence level')
        ax.set_xlim(-0.03, 1.03)
        ax.set_ylim(-0.03, 1.03)
        ax.set_title(f'Calibration: {name}')

        textstr = f'ECE = {ece:.3f}, n={len(predicted)}'
        ax.text(0.6, 0.2, textstr, transform=ax.transAxes, fontsize=10)

        ax.legend()

    plt.tight_layout()
    plt.savefig(os.path.join(directory_path,
                             f'confidence_calibration_{"folds_" if for_folds else ""}{name}'),
                dpi=600, )
    plt.show()
      
    plt.close()


In [ ]:
plot_confidence_calibration(predicted_file_path_distance_cv_gpr, name='GPR', for_folds=True)
plot_confidence_calibration(predicted_file_path_distance_cv_rf, name='RF', for_folds=True)

Figure 3: Model performances, prediction accuracy and uncertainty estimates for
GPR and RF models

In [ ]:
import matplotlib.gridspec as gridspec

file_path_results_summary = os.path.join(directory_path, "test_scores.tsv")
rf_path = os.path.join(directory_path, "RF", "predicted_target_variable_test_all_None_uncertainty_paper_soil_soil_all_data.tsv")
gpr_path = os.path.join(directory_path, "GPR", "predicted_target_variable_test_padel_None_uncertainty_paper_soil_soil_all_data.tsv")

predicted_GPR = pd.read_csv(gpr_path, sep="\t")
predicted_RF = pd.read_csv(rf_path, sep="\t")


# predicted_GPR['predicted_score'] = np.sqrt(predicted_GPR['predicted_score']**2 - predicted_GPR['experimental_std']**2)
predicted_GPR['fold'] = predicted_GPR['setup_name']


df = pd.read_csv(file_path_results_summary, sep="\t")

df["feature_reduction"] = df["feature_reduction"].fillna("None")
df["model_label"] = df["regressor"] + " | " + df["feature_reduction"]

# Choose your metrics
metrics = ["R2", "MSE", "RMSE", "MAE"]

sns.set(rc={"figure.figsize": (5, 5)})
sns.set_theme(style="whitegrid")
sns.set_style("ticks")
sns.set_context(context)


fig = plt.figure(figsize=(7, 11))
gs = gridspec.GridSpec(3, 2,wspace=0.1, hspace=0.3)

textfontsize = 7

# Top subplot spanning all 3 columns
ax_big = fig.add_subplot(gs[0, :])

# Bottom row: three separate subplots
ax1 = fig.add_subplot(gs[1, 0])
ax2 = fig.add_subplot(gs[1, 1])
ax3 = fig.add_subplot(gs[2, 0])
ax4 = fig.add_subplot(gs[2, 1])


ax = sns.boxplot(
    data=df,
    x="descriptors",
    y="R2",
    hue="model_label",
    palette=color_palette_1,
    width=0.6,  # narrower boxes = more space between groups
    showfliers=False,
    ax=ax_big
)
sns.stripplot(
    data=df,
    x="descriptors",
    y="R2",
    hue="model_label",
    dodge=True,
    palette='dark:black',
    size=3,
    alpha=0.5, legend=False,
    ax=ax_big
)

ax_big.set_xlabel("")
ax_big.set_ylabel("R2")
ax_big.set_xticklabels(ax_big.get_xticklabels(), rotation=45)
ax_big.legend(
    # bbox_to_anchor=(1, 0.35),
    
    title="Model | Feature Reduction"
)

def plot_parity_plot(predicted, axis, color, set_label=True):

    sns.scatterplot(x='experimental', y='predicted', data=predicted, alpha=0.5, s=10, ax=axis, color=color,)

    # Add +/-1 log unit lines
    min_val = min(predicted['experimental'].min(), predicted['predicted'].min()) * 0.9
    max_val = max(predicted['experimental'].max(), predicted['predicted'].max()) * 1.1

    axis.plot([min_val, max_val], [min_val, max_val], ls='-', color='black')
    axis.plot([min_val + 1, max_val], [min_val, max_val - 1], ls='--', color='black')
    axis.plot([min_val, max_val - 1], [min_val + 1, max_val], ls='--', color='black')
    axis.set(xlim=[min_val, max_val], ylim=[min_val, max_val])

    if set_label: 
        axis.set_ylabel('Predicted $\mathrm{\log DT_{50}}$') 
    else: 
        axis.set_ylabel('')
        axis.set_yticks([])

    axis.set_xlabel('Experimental $\mathrm{\log DT_{50}}$')

    # average over all folds
    r2_all = r2_score(predicted['experimental'], predicted['predicted'])
    rmse_all = root_mean_squared_error(predicted['experimental'], predicted['predicted'])

    # add a line for average

    textstr = f'R2 = {r2_all:.2f}, RMSE = {rmse_all:.2f}, n = {len(predicted)}\n'

    # add text box to plot
    axis.text(0.95, 0.05, textstr,
        fontsize=textfontsize, verticalalignment='bottom', horizontalalignment='right',
        transform=axis.transAxes)
    

def plot_uncertainty_distribution_big_plot(predicted, axis, y_min,y_max, color, set_label=True):

    sns.histplot(x='predicted_score', data=predicted, kde=True, stat="count", common_norm=False, ax=axis, color=color, binwidth=0.01)
    axis.set_xlabel('Predicted Uncertainty')
    axis.set_ylabel('Count')
    axis.set_xlim(x_min-0.1, x_max+0.1)

    average_uncertainty = np.mean(predicted['predicted_score'])
    textstr = f'Avg. Pred. Unc. = {average_uncertainty:.2f}\n'

    if set_label:
        axis.set_ylabel('Count')
    else:
        axis.set_ylabel('')
        axis.set_yticks([])

    axis.set_xlabel('Predicted Uncertainty ')
    # add text box to plot
    axis.text(0.95, 0.3, textstr,
        fontsize=textfontsize, verticalalignment='bottom', horizontalalignment='right',
        transform=axis.transAxes)

    
plot_parity_plot(predicted_GPR, ax1,color=color_palette_1[1], set_label=True)
plot_parity_plot(predicted_RF, ax2,color=color_palette_1[2], set_label=False)

x_min = min(predicted_GPR['predicted_score'].min(), predicted_RF['predicted_score'].min())
x_max = max(predicted_GPR['predicted_score'].max(), predicted_RF['predicted_score'].max())

plot_uncertainty_distribution_big_plot(predicted_GPR, ax3, x_min, x_max, color=color_palette_1[1], set_label=True)
plot_uncertainty_distribution_big_plot(predicted_RF, ax4, x_min, x_max, color=color_palette_1[2], set_label=False)
plt.tight_layout()
fig.subplots_adjust(top=0.92)
plt.savefig(os.path.join(directory_path, f"model_performance_r2_uncertainty_paper_test.pdf"), dpi=600)
plt.show()

### Application 1: DT50 prediction for TPs
Load data

In [106]:
directory_prediction = os.path.join("..", "..", "pepper_data", "predict")
directory_data = os.path.join("..", "..", "pepper_data", "data_structure")
# Training compounds with node depths
cpd_data_soil_path = os.path.join("..", "..", "pepper", "data", "soil", "cpd_data_soil_all_data.tsv")
cpd_data_soil = pd.read_csv(cpd_data_soil_path, sep="\t")

# Predicted TPs
TP_path_file = os.path.join(directory_prediction, "output", 'Predictions_pesticide_TPs.csv')
predicted_TP = pd.read_csv(TP_path_file)

# All TPs with minormajor information
TPs_with_minormajor_path = os.path.join("..", "..", "pepper", "data", "soil", "TPs", "TPs_with_minormajor.tsv")
TPs_with_minormajor = pd.read_csv(TPs_with_minormajor_path, sep="\t")


Create Figure S17: Distribution of biotransformation half-lives for different TP importance classification

In [ ]:
from matplotlib.lines import Line2D

node_depth_minor_major = pd.merge(cpd_data_soil[["node_depth", "compound_name", "logDT50_mean", "logDT50_std"]],
                                  TPs_with_minormajor[["compound_name", "pathway_name", "minor_major"]], on="compound_name",
                                  how="left")
# 
node_depth_minor_major = node_depth_minor_major.drop_duplicates(subset=["compound_name"])

col = "minormajor"  # classification column

# Convert empty strings to NaN
predicted_TP[col] = predicted_TP[col].replace(" ", pd.NA)


def classify_tp(values):
    """Classify a TP (group of rows sharing a SMILES)."""
    non_nan = values.dropna()
    unique = set(non_nan)

    # Case 1: all NaN
    if len(unique) == 0:
        return "nan"

    # Case 2: all Major or all Minor, no NaNs
    if len(unique) == 1 and values.isna().sum() == 0:
        return list(unique)[0].lower()  # "major" or "minor"

    # Case 3: mixture (different labels or label+NaN)
    return "mixed"


# Apply classification per TP
tp_class = predicted_TP.groupby("SMILES")[col].apply(classify_tp)

print

# how do I assign each smiles in the group its clas

# Count number of TPs in each category
tp_counts = tp_class.value_counts()

print("Counts of TPs per category:")
print(tp_counts)

print("\nTP → category:")
print(tp_class)

# now for all training data
# filter out rows with node_depth == 0
training_data_filtered = node_depth_minor_major[node_depth_minor_major["node_depth"] != 0]

print("\nTotal number of training compounds (with node_depth > 0): ", len(training_data_filtered))

training_data_filtered["minor_major"] = training_data_filtered["minor_major"].replace(" ", pd.NA)

# count the occurrences of each value in minor_major column
training_class = training_data_filtered.groupby("compound_name")["minor_major"].apply(classify_tp)
training_counts = training_class.value_counts()
print("\nCounts of training compounds per category:")
print(training_counts)
# now for all training data TP's

TP_summary = {
    "minor": [tp_counts.get("minor", 0) + training_counts.get("minor", 0), tp_counts.get("minor", 0),
              training_counts.get("minor", 0)],
    "major": [tp_counts.get("major", 0) + training_counts.get("major", 0), tp_counts.get("major", 0),
              training_counts.get("major", 0)],
    "mixed": [tp_counts.get("mixed", 0) + training_counts.get("mixed", 0), tp_counts.get("mixed", 0),
              training_counts.get("mixed", 0)],
    "no info": [tp_counts.get("nan", 0) + training_counts.get("nan", 0), tp_counts.get("nan", 0),
                training_counts.get("nan", 0)],
    "total": [len(predicted_TP["SMILES"].unique()) + len(training_data_filtered["compound_name"].unique()),
              len(predicted_TP["SMILES"].unique()), len(training_data_filtered["compound_name"].unique())]}

TP_summary = pd.DataFrame(TP_summary,
                          index=["total TPs", "no reported $\mathrm{DT_{50}}$", "reported $\mathrm{DT_{50}}$"])
TP_summary.to_latex(os.path.join(directory_prediction, "output", "TP_minor_major_summary.tex"))

predicted_TP["class"] = predicted_TP["SMILES"].map(tp_class)
# 
predicted_TP["reported_DT50"] = "not reported"

training_data_filtered["class"] = training_data_filtered["compound_name"].map(training_class)
training_data_filtered["reported_DT50"] = "reported"

all_TPs = pd.concat([predicted_TP[["compound_name", "pathway", "logDT50_mean_predicted", "logDT50_std_predicted",
                                   "reported_DT50", "class"]].rename(
    columns={"logDT50_mean_predicted": "logDT50", "logDT50_std_predicted": "logDT50_std"}),
                     training_data_filtered[
                         ["compound_name", "pathway_name", "logDT50_mean", "logDT50_std", "reported_DT50",
                          "class"]].rename(
                         columns={"logDT50_mean": "logDT50", "logDT50_std": "logDT50_std", "pathway_name": "pathway"})],
                    ignore_index=True)

fig, ax = plt.subplots(figsize=(6, 10))

sns.boxplot(
    data=all_TPs,
    x="class",
    y="logDT50",
    order=["major", "minor", "mixed", "nan"],
    width=0.4,
    showcaps=True,
    boxprops={"facecolor": "white", "edgecolor": "black"},
    whiskerprops={"linewidth": 1},
    medianprops={"color": "black"},
    showfliers=False,
    ax=ax
)

sns.stripplot(
    data=all_TPs[all_TPs["reported_DT50"] == "reported"],
    x="class",
    y="logDT50",
    order=["major", "minor", "mixed", "nan"],
    jitter=True,
    marker="o",
    color="black",
    size=4,
    alpha=0.9,
    ax=ax
)

sns.stripplot(
    data=all_TPs[all_TPs["reported_DT50"] == "not reported"],
    x="class",
    y="logDT50",
    order=["major", "minor", "mixed", "nan"],
    jitter=0.4,
    marker="X",
    color="tab:orange",
    size=4,
    alpha=0.9,
    hue="logDT50_std",
    palette="coolwarm",
    ax=ax
)

ax.set_xlabel("")
ax.set_ylabel(r"log DT$_{50}$")

# move legend out of the plot
custom_handles = [
    Line2D([0], [0], marker='o', color='none', markerfacecolor='black',
           markersize=7, label='reported', markeredgewidth=0.5,
           linestyle='None', ),
    Line2D([0], [0], marker='X', color='none', markerfacecolor='grey',
           markersize=7, label='predicted', markeredgewidth=0.5,
           linestyle='None', )
]
# add marker legend
marker_legend = ax.legend(handles=custom_handles, title="Reported DT50",
                          bbox_to_anchor=(1.05, 1), loc="upper left")
ax.add_artist(marker_legend)

# now auto-legend from seaborn for hue
ax.legend(title="logDT50_std", bbox_to_anchor=(1.05, 0.9))

plt.savefig(os.path.join(directory_prediction, "output", "TP_minor_major_boxplot.pdf"), dpi=600, bbox_inches="tight")
plt.show()

# print for the minor_reported dt50 class the highest two dt50 mean values with their std
minor_reported = all_TPs[(all_TPs["class"] == "minor") & (all_TPs["reported_DT50"] == "reported")]
top2_minor = minor_reported.nlargest(2, "logDT50")
print("Top 2 minor reported DT50 compounds:")
print(top2_minor)


def smiles_to_img_array(smile, size=(400, 400)):
    """Return a NumPy array with a RDKit drawing of the SMILES."""
    mol = Chem.MolFromSmiles(smile)
    drawer = rdMolDraw2D.MolDraw2DCairo(size[0], size[1])
    opts = drawer.drawOptions()
    opts.useBWAtomPalette()
    drawer.DrawMolecule(mol)
    drawer.FinishDrawing()
    png_bytes = drawer.GetDrawingText()
    img = Image.open(io.BytesIO(png_bytes))
    return np.array(img)


# create figure with two sublopts: left structure on and right strutcture two, in between I want text boxes with the compound name, logDT50 value and uncertainty
sns.set(rc={"figure.figsize": (8, 4)})
sns.set_theme(style="whitegrid")
sns.set_style("ticks")
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(8, 4))
smile_1 = "NS(=O)(=O)c1ncccc1C(F)(F)F"
smile_2 = "CC(C)NC(=O)NC(=O)Nc1cc(Cl)cc(Cl)c1"

img_array_1 = smiles_to_img_array(smile_1)
img_array_2 = smiles_to_img_array(smile_2)

ax1.imshow(img_array_1)
ax1.axis('off')
ax2.imshow(img_array_2)
ax2.axis('off')
plt.tight_layout()
plt.savefig(os.path.join(directory_prediction, "output", "compound_structures.pdf"), dpi=600)
plt.show()

# save images as separate files as png
img1 = Image.fromarray(img_array_1)
img2 = Image.fromarray(img_array_2)
img1.save(os.path.join(directory_prediction, "output", "compound_1_structure.png"))
img2.save(os.path.join(directory_prediction, "output", "compound_2_structure.png"))

### Application 2: DT50 prediction for ZeroPM Substances

Define plotting functions

In [130]:
# Functions to plot ZeroPM predictions
edgecolor = 'None'
markersize = 5

# color_map - from COMPASS
def get_color_class_mapping(df_reference_tsne):
    # This is the palette I used in my publication for the top 15 most common ClassyFire Superclasses + some code snippets
    palette = ['darkslategrey', 'teal', 'aquamarine', 'darkred', 'orangered', 'mediumpurple', 'darkorchid',
               'mediumblue', 'royalblue', 'skyblue', 'darkgoldenrod', 'darkorange', 'gold', 'lightpink',
               'hotpink',
               'lightgrey']

    top15 = df_reference_tsne.groupby('Superclass').count()['TSNE1'].sort_values(ascending=False).index[:15]
    df_reference_tsne['Superclass (top 15)'] = df_reference_tsne['Superclass'].where(
        df_reference_tsne['Superclass'].isin(top15), 'Other')
    hue_order = top15.sort_values().to_list() + ['Other']
    return palette, top15, hue_order, df_reference_tsne

def figure_4():
    fig = plt.figure(figsize=(13, 16.5), layout='constrained')

    subfigs = fig.subfigures(2, 1, height_ratios=[1, 2])

    axsBottom = subfigs[1].subplots(2, 2, sharey=True, sharex=True)
    axleftup = axsBottom[0][0]
    axrightup = axsBottom[0][1]
    axleftdown = axsBottom[1][0]
    axrightdown = axsBottom[1][1]

    axsTopAll = subfigs[0].subplots(1, 2, gridspec_kw={'width_ratios': [7, 1]})
    axsTop = axsTopAll[0]
    axsTopAll[1].remove()

    sns.set_context('paper', font_scale=1.5)

    # top
    sns.scatterplot(x='TSNE1', y='TSNE2', data=df_reference, ax=axsTop, color='lightgrey', alpha=0.1,
                    edgecolor=edgecolor, s=markersize)
    sns.scatterplot(x='TSNE1', y='TSNE2', data=df_reference, ax=axsTop, hue='Superclass (top 15)',
                    palette=palette, edgecolor=edgecolor, s=5)
    sns.move_legend(axsTop, "upper left", bbox_to_anchor=(1.146, 1))

    # bottom
    # custom_palette = sns.diverging_palette(220, 20, as_cmap=True, center='dark')
    sns.scatterplot(x='TSNE1', y='TSNE2', data=all_data, ax=axrightup, color='lightgrey', alpha=0.1,
                    linestyle=edgecolor, s=markersize)
    sns.scatterplot(x='TSNE1', y='TSNE2', data=predictions, ax=axrightup, hue=r'${logDT}_{50,std}$',
                    palette="vlag", edgecolor=edgecolor, s=5)

    sns.scatterplot(x='TSNE1', y='TSNE2', data=all_data, ax=axrightdown, color='lightgrey', alpha=0.1,
                    linestyle=edgecolor, s=markersize)
    sns.scatterplot(x='TSNE1', y='TSNE2', data=predictions, ax=axrightdown, hue='p(P)', palette="magma_r",
                    edgecolor=edgecolor, s=5)

    # bottom
    sns.scatterplot(x='TSNE1', y='TSNE2', data=all_data, ax=axleftup, color='lightgrey', alpha=0.1,
                    linestyle=edgecolor, s=markersize)
    sns.scatterplot(x='TSNE1', y='TSNE2', data=predictions, ax=axleftup, hue=r'${logDT}_{50,mean}$',
                    palette="Spectral_r", edgecolor=edgecolor, s=5)

    sns.scatterplot(x='TSNE1', y='TSNE2', data=all_data, ax=axleftdown, color='lightgrey', alpha=0.1,
                    linestyle=edgecolor, s=markersize)
    sns.scatterplot(x='TSNE1', y='TSNE2', data=predictions, ax=axleftdown, hue='Training',
                    palette={True: "black", False: "lightblue"}, edgecolor=edgecolor, s=5)
    true_only = predictions[predictions['Training'] == True]
    sns.scatterplot(x='TSNE1', y='TSNE2', data=true_only, ax=axleftdown, color='black', edgecolor=edgecolor,
                    s=5, legend=False)

    plt.savefig(os.path.join(pep.get_data_directory(),
                             "predict", "output", 'chemical_space_persistence.png'), dpi=300)
    plt.savefig(os.path.join(pep.get_data_directory(),
                             "predict", "output", 'chemical_space_persistence.pdf'))
    plt.close()


# plot predicted mean vs std #
def figure_S19():
    x = r'${logDT}_{50,mean}$'
    y = r'${logDT}_{50,std}$'
    sns.set_context('paper', font_scale=1.5)
    experimental = predictions[predictions['Training'] == True]
    ax = sns.scatterplot(data=predictions, x=x, y=y, hue='p(P)', palette="icefire", legend=True)
    sns.scatterplot(data=experimental, x=x, y=y, color='black', marker='x', ax=ax, legend=False)
    plt.axvline(x=np.log10(120), label='P', linestyle=':', color='black')
    plt.axvline(x=np.log10(180), label='vP', linestyle='--', color='black')
    plt.text(2.3, 1.13, 'vP')
    plt.text(2.04, 1.13, 'P', horizontalalignment='right')

    plt.tight_layout()
    plt.savefig(os.path.join(pep.get_data_directory(),
                             "predict", "output", 'scatterplot_mean_std_predicted.png'), dpi=300)
    plt.savefig(os.path.join(pep.get_data_directory(),
                             "predict", "output", 'scatterplot_mean_std_predicted.pdf'))
    plt.close()


def print_top_persistent(p=0.7):
    subset = predictions[predictions['p(P)'] >= p]
    subset = subset[subset['logDT50_mean_experimental'].isna()]
    subset.to_csv(os.path.join(pep.get_data_directory(),
                             "predict", "output", 'top_persistent.tsv'), sep='\t', index=False)
    print(len(subset))


def print_top_percent_stats(percent=0.08):
    top_x = int(percent * len(all_data))
    print(f"8 percent corresponds to {top_x} substances")
    sorted_predictions = predictions.sort_values(by=r'${logDT}_{50,std}$', ascending=True)
    subset = sorted_predictions.loc[:][:top_x]
    average_std = np.mean(subset[r'${logDT}_{50,std}$'])
    print("average std for top 8 %:", average_std, ', max:', np.max(subset[r'${logDT}_{50,std}$']))
    stats = subset.describe()
    print(stats)  # max is 0.76

Load and analyze ZeroPM predictions

In [ ]:
# reference data and coordinates were obtained from COMPASS
# the coordinates data file can be downloaded from Zenodo: https://doi.org/10.5281/zenodo.18723920
# and saved under pepper_data/soil_paper_output
coordinates = pd.read_csv(os.path.join(save_output_path, "output_zeropm_partial.csv"))

coordinates.drop_duplicates(subset=['SMILES'], keep='first', inplace=True)
# colors for superclasses
palette, top15, hue_order, df_reference = get_color_class_mapping(coordinates)

# file with ZeroPM predictions can be found in output
path_to_zeropm_predictions = os.path.join(pep.get_data_directory(),
                                          "predict", "output", "filtered_zeropm_smiles__all_data.tsv")
df_pred = pd.read_csv(path_to_zeropm_predictions, sep="\t")

df_pred.rename(columns={'SMILES': 'new_SMILES'}, inplace=True)
df_pred.rename(columns={'original_SMILES': 'SMILES'}, inplace=True)

# mapp predictions back to reference coordinates
all_data = df_reference.merge(df_pred, how='left', on='SMILES')

all_data.to_csv(os.path.join(pep.get_data_directory(),
                "predict", "output", 'all_zeropm_predictions_with_coordinates.csv'), index=False)

# new data frame with only predicted values
predicted_only = all_data.dropna(subset=['logDT50_mean_predicted'])
predicted_only['Training'] = np.where(np.isnan(predicted_only['logDT50_mean_experimental']), False, True)
print('Predictions:', len(predicted_only))
predictions = predicted_only.rename(columns={'logDT50_mean_predicted': 'predicted mean logDT50',
                                             'logDT50_std_predicted': r'${logDT}_{50,std}$',
                                             'logDT50_mean_predicted': r'${logDT}_{50,mean}$',
                                             'Persistent': 'p(P)', 'non-Persistent': 'p(nP)'})

Create Figures 4 and S19

In [ ]:
figure_4()
figure_S19()
print_top_persistent()
print_top_percent_stats()

### Comparison ZeroPM predictions with TP predictions

In [ ]:
TP_prediction_path = os.path.join(directory_prediction, "output", "final_predictions_soil_without_TP_raw.tsv")
zeropm_prediction_path = os.path.join(directory_prediction, "output", "all_zeropm_predictions_with_coordinates.csv")

predicted_TP = pd.read_csv(TP_prediction_path, sep="\t")
predicted_zeropm = pd.read_csv(zeropm_prediction_path)

def hist_plot_prediction_std(predicted, title): 
    predicted.dropna(subset=['logDT50_std_predicted'], inplace=True)
    # filter out predicted std <0.4
    predicted = predicted[predicted['logDT50_std_predicted'] >= 0.4]

    print(f"average predicted std for {title}: ", predicted['logDT50_std_predicted'].mean())

    ax = sns.histplot(x='logDT50_std_predicted', data=predicted, kde=True, stat="count", common_norm=False)
    ax.set_xlabel('Predicted Uncertainty')
    ax.set_ylabel('Count')
    ax.set_title(title)
    plt.show()
    plt.close()

hist_plot_prediction_std(predicted_TP, title="TP Predictions Uncertainty Distribution")
hist_plot_prediction_std(predicted_zeropm, title="ZeroPM Predictions Uncertainty Distribution")

### S4.2 Validation on external dataset

Load benchmarking data: 99 reported half-lives for 25 compounds

In [156]:
reference_data_path = os.path.join('..', 'data', 'soil', 'benchmarking', 'cpd_data_soil_25.tsv')
reference_data = pd.read_csv(reference_data_path, sep='\t')

Load PEPPER predictions

In [157]:
pepper_predictions_path = os.path.join('..', 'data', 'soil', 'benchmarking', 'predictions_pepper.tsv')
pepper_predictions = pd.read_csv(pepper_predictions_path, sep='\t')
for column in pepper_predictions.columns:
    if column == 'ID': continue
    pepper_predictions.rename(columns={column: f'{column}_pepper'}, inplace=True)

Load VEGA predictions (calculated with VEGA standalone version 1.2.6 on 26.02.2026)

In [158]:
vega_predictions_path = os.path.join('..', 'data', 'soil', 'benchmarking', 'predictions_vega.tsv')
vega_predictions = pd.read_csv(vega_predictions_path, sep='\t')
vega_predictions.rename(columns={'Persistence (soil) quantitative model (IRFMN)-prediction [log(days)]': 'logDT50_VEGA'}, inplace=True)

Load BIOWIN4 predictions

In [159]:
biowin_predictions_path = os.path.join('..', 'data', 'soil', 'benchmarking', 'predictions_biowin.tsv')
biowin_predictions = pd.read_csv(biowin_predictions_path, sep='\t')

# function to transform biowin output to half-lives (Arnot et al.)
def arnot_transform_biowin4(biowin):
    hl = (-1.46 * biowin + 6.51) #+ np.log10(2)
    return hl
# biowin_predictions['logDT50_BIOWIN4'] = biowin_predictions['BIOWIN4'].apply(lambda x: arnot_transform_biowin4(x))
biowin_predictions['logDT50_BIOWIN4'] = biowin_predictions['BIOWIN4'].apply(arnot_transform_biowin4)

Merge data into one dataframe

In [160]:
data = reference_data.merge(pepper_predictions, how='left', on='ID')
data = data.merge(biowin_predictions, how='left', on='ID')
data = data.merge(vega_predictions, how='left', on='ID') 

Print statistics

In [161]:
data.dropna(axis='rows', subset='logDT50_mean_predicted_pepper', inplace=True)
print('Pepper RMSE and R2')
rmse_pepper = root_mean_squared_error(data['logDT50_mean'], data['logDT50_mean_predicted_pepper'])
r2_pepper = r2_score(data['logDT50_mean'], data['logDT50_mean_predicted_pepper'])
print(rmse_pepper, r2_pepper)

print('Biowin RMSE and R2')
rmse_biowin = root_mean_squared_error(data['logDT50_mean'], data['logDT50_BIOWIN4'])
r2_biowin = r2_score(data['logDT50_mean'], data['logDT50_BIOWIN4'])
print(rmse_biowin, r2_biowin)

print('VEGA RMSE and R2')
rmse_vega = root_mean_squared_error(data['logDT50_mean'], data['logDT50_VEGA'])
r2_vega = r2_score(data['logDT50_mean'], data['logDT50_VEGA'])
print(rmse_vega, r2_vega)

Pepper RMSE and R2
1.4583842556493076 0.2810394967127273
Biowin RMSE and R2
1.5758227862687606 0.1605866913808328
VEGA RMSE and R2
1.6898615327890034 0.03469770570268105


Define helper functions for the plot:

In [162]:
def add_lines(ax, min_val = -2.5, max_val = 4.5):
    ax.set_ylim(min_val, max_val)
    ax.set_xlim(min_val, max_val)
    ax.plot([min_val, max_val], [min_val, max_val], ls='-', color='black')
    ax.plot([min_val + 1, max_val], [min_val, max_val - 1], ls='--', color='black')
    ax.plot([min_val, max_val - 1], [min_val + 1, max_val], ls='--', color='black')
    return ax

def add_performance(ax, r2, rmse):
    ax.text(-2.4, 4.2, 'R2: {}'.format(round(r2, 2)))
    ax.text(-2.4, 3.8, 'RMSE: {}'.format(round(rmse, 2)))
    return ax

Plot figure with three panels for pepper, BIOWIN, and VEGA prediction evaluation (Figure S13)

In [163]:
fig, ax = plt.subplots(1, 3, figsize=(12, 4))
palette = sns.color_palette('husl', 3)
sns.set_style('white')
sns.set_context('paper')

# add pepper
data['logDT50_PEPPER'] = data['logDT50_mean_predicted_pepper'] # rename column
sns.scatterplot(x='logDT50_mean', y='logDT50_PEPPER', data=data, 
                     hue='source', palette=palette, ax=ax[0]
)
ax[0] = add_lines(ax[0])
ax[0] = add_performance(ax[0], r2_pepper, rmse_pepper)

ax[0].errorbar(x=data['logDT50_mean'], y=data['logDT50_PEPPER'],
             xerr=data['logDT50_std']*1.96, yerr=data['logDT50_std_predicted_pepper']*1.96, fmt='.', alpha=0.2,
              color='black', elinewidth = 0.5)

# add biowin
sns.scatterplot(x='logDT50_mean', y='logDT50_BIOWIN4', data=data, 
                     hue='source', palette=palette, ax=ax[1]
)
ax[1] = add_lines(ax[1])
ax[1] = add_performance(ax[1], r2_biowin, rmse_biowin)
ax[1].errorbar(x=data['logDT50_mean'],y=data['logDT50_BIOWIN4'],
             xerr=data['logDT50_std']*1.96, fmt='.', alpha=0.2,
              color='black', elinewidth = 0.5)

# add vega
sns.scatterplot(x='logDT50_mean', y='logDT50_VEGA', data=data, 
                     hue='source', palette=palette, ax=ax[2]
)
ax[2] = add_lines(ax[2])
ax[2] = add_performance(ax[2], r2_vega, rmse_vega)
ax[2].errorbar(x=data['logDT50_mean'],y=data['logDT50_VEGA'],
             xerr=data['logDT50_std']*1.96, fmt='.', alpha=0.2,
              color='black', elinewidth = 0.5)



plt.tight_layout()
plt.savefig(os.path.join(pep.get_data_directory(),'soil_paper_output', 'SI_scatterplot_biowin_vs_vega_vs_pepper.pdf'))
plt.close()



Save prediction comparison as latex table

In [164]:
# save data as latex table
data_for_SI = data.loc[:,['ID','compound_name', 'source', 'logDT50_mean', 'logDT50_std', 
                      'logDT50_PEPPER', 'logDT50_BIOWIN4', 'logDT50_VEGA', 'logDT50_std_predicted_pepper']]
data_for_SI.rename(columns = {'logDT50_std_predicted_pepper': 'PEPPER_uncert',
                              'compound_name': 'Compound name',
                              'logDT50_mean': 'logDT50 mean',
                              'logDT50_std': 'logDT50 std',
                              'logDT50_PEPPER': 'PEPPER',
                              'logDT50_VEGA': 'VEGA',
                              'logDT50_BIOWIN4': 'BIOWIN4',
                              }, inplace = True)
data_for_SI.to_latex(buf=os.path.join(pep.get_data_directory(),'soil_paper_output', 'SI_table_benchmark.tex'),
                     index=False, float_format="{:0.2f}".format)